In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
here = Path.cwd()
datadir = here.parent/"Data_CSVs/"
#print(datadir)
datadirtemp = here.parent/"SimulationScripts/tempXYZ/"
#print(datadirtemp)
cmap = plt.cm.get_cmap('BuPu')
clrs=["#c7e9b4","#7fcdbb","#41b6c4","#1d91c0","#225ea8","#0c2c84","#c7e9b4","#7fcdbb","#41b6c4","#1d91c0","#225ea8","#0c2c84"]
clrs = ['#bfd3e6','#9ebcda','#8c96c6','#8c6bb1','#88419d','#6e016b','#bfd3e6','#9ebcda','#8c96c6','#8c6bb1','#88419d','#6e016b']


Some simulation definitions:

In [4]:
def InitialPositions(num_particles, box_size, sigma, cluster_size):
    """Generate clustered initial positions in a 2D box, avoiding overlaps."""
    positions = []
    placed_particles = 0
    cluster_centers = []

    num_clusters = max(1, num_particles // cluster_size)

    for _ in range(num_clusters):
        center = np.random.uniform(-box_size / 2 + 3*sigma, box_size / 2 - 3*sigma, size=2)
        cluster_centers.append(center)

    for center in cluster_centers:
        # Calculate how many rows/cols we need
        particles_to_place = min(cluster_size, num_particles - placed_particles)
        grid_dim = int(np.ceil(np.sqrt(particles_to_place)))
        spacing = sigma * 1.1  # Slight buffer to avoid touching

        # Create a grid centered on (0,0), then shift to cluster center
        for i in range(grid_dim):
            for j in range(grid_dim):
                if placed_particles >= num_particles or (i * grid_dim + j) >= particles_to_place:
                    break
                offset = np.array([(i - grid_dim // 2) * spacing,
                                   (j - grid_dim // 2) * spacing])
                candidate = center + offset

                # Stay inside the box
                if (-box_size/2 + sigma < candidate[0] < box_size/2 - sigma and
                    -box_size/2 + sigma < candidate[1] < box_size/2 - sigma):
                    positions.append(candidate)
                    placed_particles += 1

    return np.array(positions)
    
def parse_lammps_xyz(filename):
    data = []
    with open(filename, 'r') as file:
        while True:
            line = file.readline()
            if not line:
                break  # EOF
            
            if "ITEM: TIMESTEP" in line:
                timestep = int(file.readline().strip())
                file.readline()  # Skip "ITEM: NUMBER OF ATOMS"
                num_atoms = int(file.readline().strip())
                file.readline()  # Skip "ITEM: BOX BOUNDS ..."
                file.readline()
                file.readline()
                file.readline()
                file.readline()  # Skip "ITEM: ATOMS id type x y z"
                
                for _ in range(num_atoms):
                    atom_line = file.readline()
                    if not atom_line:
                        break  # Unexpected EOF
                    parts = atom_line.strip().split()
                    atom_id, atom_type = int(parts[0]), int(parts[1])
                    x, y, z = map(float, parts[2:5])
                    data.append({
                        "timestep": timestep,
                        "id": atom_id,
                        "type": atom_type,
                        "x": x,
                        "y": y,
                        "z": z
                    })

    df = pd.DataFrame(data)
    return df

# Unwrapping the trajectories
# The function unwrap_trajectories takes a DataFrame with atom trajectories and unwraps them based on periodic boundary conditions.

def unwrap_trajectories(df, box_length_x, box_length_y):
    # Sort the DataFrame by atom ID and timestep
    df = df.sort_values(by=["id", "timestep"]).copy()
    unwrapped_rows = []

    for atom_id, group in df.groupby("id"):
        group = group.sort_values(by="timestep").copy()
        x = group["x"].values
        y = group["y"].values
        timesteps = group["timestep"].values
        unwrappedX = np.zeros_like(x)
        unwrappedY = np.zeros_like(y)
        #unwrappedX = [x[0]]
        #unwrappedY = [y[0]]
        unwrappedX[0] = x[0]
        unwrappedY[0] = y[0]
        
        shiftX = 0.0
        shiftY = 0.0

        for i in range(1, len(x)):
            dx = x[i] - x[i - 1]
            dy = y[i] - y[i - 1]
            if dx > box_length_x / 2:
                #print('dx > box_length_x / 2')
                shiftX -= box_length_x
            if dy > box_length_y / 2:
                #print('dy > box_length_y / 2')
                shiftY -= box_length_y
            if dx < -box_length_x / 2:
                #print('dx < -box_length_x / 2')
                shiftX += box_length_x
            if dy < -box_length_y / 2:
                #print('dy < -box_length_y / 2')
                shiftY += box_length_y
            #print('i',i)
            #print('len(UnwrappedX)',len(unwrappedX))
            unwrappedX[i] = x[i] + shiftX
            unwrappedY[i] = y[i] + shiftY
            # if shiftX != 0 or shiftY != 0:
            #     print(f"Atom {atom_id} at timestep {timesteps[i]} shifted by ({shiftX}, {shiftY})")
            #     print("box_size_x", box_length_x)

        group["x_unwrapped"] = unwrappedX
        group["y_unwrapped"] = unwrappedY
        unwrapped_rows.append(group)

    return pd.concat(unwrapped_rows, ignore_index=True) #df

# Function to write positions to .xyz file
def write_xyz(f, positions, step,box_length):
    #with open(f, 'a') as f:
    f.write("ITEM: TIMESTEP\n")
    f.write(f"{step}\n")
    f.write("ITEM: NUMBER OF ATOMS\n")
    f.write(f"{len(positions)}\n")
    # Write box dimensions (optional)
    f.write("ITEM: BOX BOUNDS pp pp pp\n")
    f.write(f"{-box_length/2} {box_length/2}\n")
    f.write(f"{-box_length/2} {box_length/2}\n")
    f.write(f"0.0 0.1\n")
    f.write("ITEM: ATOMS id type x y z\n")
    i=0
    for pos in positions:
        i+=1
        f.write(f"{i} 1 {pos[0]} {pos[1]} 0.0\n") 
        #f.write(f"1 {pos[0]} {pos[1]} 0.0\n")  # Assuming particles are Argon atoms



Simulation itself:

In [5]:
# write a lammps config file and input file
## Standard Parameters
frames = [1,3,10,20,30,40,50] #[3,10,30] #3,10,30,100,300,1000 #1,3,10,20,
numframes_s  = [300]
temperatures = 0.02 #,1.0,2.0,5.0
num_particles_s = [3]
#folder = 'FrameSweep2'
folder = 'Simulations/LimitsLibrary' #'FrameSweep' #ParticleNumber' #
outdir = datadir / folder
outdir.mkdir(parents=True, exist_ok=True)   # create if missing

numequiframes = 5000
epsilon = 0.0
sigma = 1.5 #1.25 #2.5 #+ee*0.5
rcut=0.1 #sigma*2 #1.22 # repulsion only
l_da = 0.1 # Softening parameter for the Lennard-Jones potential
cluster_size= 5
num_particles = 200
kappa = 0.01 # Spring constant for the harmonic trap
Center = np.array([0,0]) #Center of harmonic trap
pair_interactions = True # Set to True to enable Lennard-Jones pair interactions
box_size = 50.0  # Box is a cube of size box_size x box_size
temperature = 0.1 #1.0 #1.0 #0.2 #0.2  # Temperature (k_B * T)
dt = 0.001  # Time step
damp = 1 #0.1
frame = 20
Frac_Delete = 0.0 #1
#Num_Deleted = int(num_particles*Frac_Delete)
Overlap = 0.9
#frame = 10
#new parameters:
Ncycles = 20

Np = num_particles
numframes = 1000 #numframes_s[0]
#for frame in frames:
#for numframes in numframes_s:
InitialPositions_xyz = InitialPositions(Np, box_size, sigma, cluster_size)

num_particles = InitialPositions_xyz.shape[0]
print("num_particles",num_particles)

seed = 1
#num_particles = 17

print("starting loop for frame",frame)
num_steps = numframes*frame #00
#t_steps_delete = frames_to_delete*frame
equilibration_steps = numequiframes*frame #00
#name0 = "outputAll_kappa"+str(kappa)+"_temp"+str(temperature)+"_trajlen"+str(numframes)+"_dt"+str(dt)+"_N"+str(num_particles)+"_seed"+str(seed)+"_damp"+str(damp)+"frame"+str(frame)+".xyz"
#name0 = "outputVariableNodes_kappa"+str(kappa)+"_temp"+str(temperature)+"_trajlen"+str(numframes)+"_dt"+str(dt)+"_N"+str(num_particles)+"_seed"+str(seed)+"_damp"+str(damp)+"frame"+str(frame)+".xyz"
name0 = "outputTestingGamma_NoTrap_kappa"+str(kappa)+"_temp"+str(temperature)+"_trajlen"+str(numframes)+"_dt"+str(dt)+"_N"+str(num_particles)+"_seed"+str(seed)+"_damp"+str(damp)+"frame"+str(frame)+".xyz"
name = "/"+name0 #outputNoise_temp"+str(temperature)+"_dt"+str(dt)+"_N"+str(num_particles)+"_seed"+str(seed)+"_damp"+str(damp)+"frame"+str(frame)+".xyz"
print("name",name)

print("writing config file")

with open(datadirtemp / "config.in", "w") as f:
    molid = 0
    f.write('Configuration file Under damped cells in clusters\n')
    f.write(str(num_particles)+' atoms\n')
    f.write('''0 bonds
0 angles
2 atom types

''')
    f.write(str(-box_size/2)+' '+str(box_size/2)+' xlo xhi\n')
    f.write(str(-box_size/2)+' '+str(box_size/2)+' ylo yhi\n')
    f.write(str(-0.1)+' '+str(0.1)+' zlo zhi\n')
    f.write('''
Masses

1 1.0
2 1.0
''')
    f.write('''

Atoms

''')
    for n in range(num_particles):
        molid +=1
        x,y =  InitialPositions_xyz[n] 
        z=0
        if np.random.rand() < Frac_Delete:
            f.write(str(molid)+' 2 '+str(x)+' '+str(y)+' '+str(z)+'\n')
        else:
            f.write(str(molid)+' 1 '+str(x)+' '+str(y)+' '+str(z)+'\n')

    f.close
print("writing input file")
with open(datadirtemp / "input.in", "w") as f:
    f.write(f"units lj\n")
    f.write(f"atom_style atomic\n")
    f.write(f"dimension 2\n")
    f.write(f"boundary p p p\n")
    f.write(f"read_data {datadirtemp}/config.in\n")
    f.write(f"timestep {dt}\n")
    f.write(f"variable            CenteringForceX atom -{kappa*2}*x\n")
    f.write(f"variable            CenteringForceY atom -{kappa}*y\n")
    f.write(f"pair_style lj/cut/soft 1 0.5 {rcut}\n")
    f.write(f"pair_coeff * * {epsilon} {sigma} {l_da} {rcut}\n")
    f.write(f"velocity all create {temperature} 12345\n")
    f.write(f"group TheseAreDeleted type 2\n")
    f.write(f"region CreateAtomsHere block -2 2 -2 2 -0.1 0.1\n")
    f.write(f"dump 1 all custom {frame} {datadirtemp}{name} id type x y z\n")
    f.write(f"fix centering all addforce v_CenteringForceX v_CenteringForceY 0.0 every 1\n")
    f.write(f"fix 2 all langevin {temperature} {temperature} {damp} 12345\n")
    f.write(f"fix 1 all nve\n")
    f.write(f"run {equilibration_steps}\n")
    f.write(f"unfix centering\n")
    #f.write(f"dump 1 all custom {frame} {datadirtemp}{name} id type x y z\n")
    f.write(f"run {num_steps}\n")
    
    f.close()
    
inputfile =datadirtemp/"input.in"
logfile = datadirtemp/"log.lammps"


!/Users/billiemeadowcroft/Dropbox/PostDocCode/lammps1/src/lmp_serial -in {inputfile} -log {logfile}
print()
print("finished running lammps")

df_atoms = parse_lammps_xyz(datadirtemp / name0)
print(name)
#box_size = 100.0
data_unwrapped = unwrap_trajectories(df_atoms, box_size, box_size)
data_xyz = data_unwrapped[['timestep', 'id', 'x', 'y','z']].copy()
print("DATA1",data_xyz.head())
data_xyz = data_xyz[data_xyz['timestep'] >= int(equilibration_steps)]
print("DATA1",data_xyz.head())
# rename time to start from 0
data_xyz['timestep'] = data_xyz['timestep'] - int(equilibration_steps)
print("DATA2",data_xyz.head())
# make timestep a frame index
data_xyz['timestep'] = data_xyz['timestep']/frame
print("DATA3",data_xyz.head())
# convert timestep to integer
data_xyz['timestep'] = data_xyz['timestep'].astype(int)
print("DATA4",data_xyz.head())
# re-order by time and ID 
data_xyz = data_xyz.sort_values(by=['timestep', 'id'])
    



num_particles 200
starting loop for frame 20
name /outputTestingGamma_NoTrap_kappa0.01_temp0.1_trajlen1000_dt0.001_N200_seed1_damp1frame20.xyz
writing config file
writing input file
LAMMPS (29 Aug 2024 - Update 2)
Reading data file ...
  orthogonal box = (-25 -25 -0.1) to (25 25 0.1)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  200 atoms
  read_data CPU = 0.001 seconds
0 atoms in group TheseAreDeleted
Generated 0 of 1 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 0.4
  ghost atom cutoff = 0.4
  binsize = 0.2, bins = 250 250 1
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair lj/cut/soft, perpetual
      attributes: half, newton on
      pair build: half/bin/atomonly/newton
      stencil: half/bin/2d
      bin: standard
Setting up Verlet run ...
  Unit style    : lj
  Current step  : 0
  Time step     :

In [6]:
print("DATA5",data_xyz.head())
#data_xyz = data_xyz.sort_values(by='timestep')
data_xyz.columns = ['frame', 'particle', 'x', 'y','z']
# add a column 'mass' to the dataframe with value 1.0
data_xyz['mass'] = 1.0

data_xyz.to_csv(datadir / folder /name0.replace('.xyz', '.csv'), index=False)

DATA5        timestep  id         x        y         z
5000          0   1  1.701000 -2.02762  0.015328
11001         0   2  2.503760 -0.86357  0.090781
17002         0   3  0.101883 -6.85990 -0.038869
23003         0   4  0.732967 -8.89128 -0.063746
29004         0   5 -0.220305 -3.98337 -0.001117
